<a href="https://colab.research.google.com/github/chulbulisahu2-cloud/2502110065-IT6/blob/main/Lab_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IT 12

In [1]:
import os

os.makedirs('data', exist_ok=True)

log_data = """PE-0137,25.02,DAY
PE-0201,24.90,NIGHT
PE-0345,25.04,DAY, re-gauged
INVALID_LINE_1
PE-0999,25.00,SWING
PE-0111,25.10,DAY
PE-0222,BAD_VALUE,NIGHT
PE-0333,24.85,DAY
PE-0444,24.95,NIGHT
BROKEN,DATA,LINE
PE-0555,25.01,DAY
PE-0666,25.08,SWING
PE-0777,24.99,NIGHT
PE-0888,25.00,DAY
PE-0900,24.88,NIGHT
"""

with open('data/shift_log.txt', 'w', encoding='utf-8') as f:
    f.write(log_data)

with open('data/shift_log.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"Total lines in log: {len(lines)}")

Total lines in log: 15


IT 10

In [3]:
valid_records = []
rejects = []

with open('data/shift_log.txt', 'r', encoding='utf-8') as f:
    for line in f:
        line_str = line.strip()
        if not line_str:
            continue
        parts = [p.strip() for p in line_str.split(',')]

        # Basic validation check: expect 3 main fields (Part Code, Value, Shift)
        if len(parts) >= 3 and parts[0].startswith('PE-'):
            try:
                val = float(parts[1])
                valid_records.append((parts[0], val, parts[2]))
            except ValueError:
                rejects.append(f"{line_str},Invalid float value")
        else:
            rejects.append(f"{line_str},Malformed row")

# Write human report
with open('data/report.txt', 'w', encoding='utf-8') as f:
    f.write("PART CODE   | VALUE   | SHIFT\n")
    f.write("-" * 30 + "\n")
    for code, val, shift in valid_records:
        f.write(f"{code:<11} | {val:<7.2f} | {shift}\n")

# Write rejects
with open('data/rejects.csv', 'w', encoding='utf-8') as f:
    f.write("Line,Reason\n")
    for r in rejects:
        f.write(f"{r}\n")

print(f"Processed: {len(valid_records)} valid, {len(rejects)} rejected.")

Processed: 12 valid, 3 rejected.


IT 8

In [5]:
import re

# Regex pattern matching PE-XXXX, float, and shift text with flexible spacing
pattern = re.compile(r'^\s*(PE-\d{4})\s*,\s*([+-]?\d*\.\d+|\d+)\s*,\s*([A-Za-z]+)')

valid_records = []
rejects = []

with open('data/shift_log.txt', 'r', encoding='utf-8') as f:
    for line in f:
        line_str = line.strip()
        if not line_str:
            continue

        match = pattern.match(line_str)
        if match:
            code, val_str, shift = match.groups()
            valid_records.append((code, float(val_str), shift))
        else:
            rejects.append(f"{line_str},Regex mismatch")

print(f"Regex Parsed Valid: {len(valid_records)}, Rejects: {len(rejects)}")

Regex Parsed Valid: 12, Rejects: 3


IT 6

In [4]:
import os
import glob
from datetime import datetime

os.makedirs('data', exist_ok=True)
history_file = 'data/run_history.csv'

# Initialize history file with header if it doesn't exist
if not os.path.exists(history_file):
    with open(history_file, 'w', encoding='utf-8') as f:
        f.write("Timestamp,FileName,Processed,Rejects\n")

txt_files = glob.glob('data/*.txt')

for file_path in txt_files:
    if 'report' in file_path:
        continue  # skip existing reports

    processed_count = 0
    reject_count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                if "PE-" in line:
                    processed_count += 1
                else:
                    reject_count += 1

    # Log run history
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    filename = os.path.basename(file_path)

    with open(history_file, 'a', encoding='utf-8') as f:
        f.write(f"{timestamp},{filename},{processed_count},{reject_count}\n")

print("Batch processing complete. Run history updated.")

Batch processing complete. Run history updated.
